# Part 1 - Many In, One Out

### Which weight changed the most over training? Which changed the least? Why?

The first weight changed the most over training as it started as 0.1, but changed to 0.116 after 10 iterations. On the other end, the last weight changed from 0.2 to 0.201 after 10 iterations.

This is because the first input of sensing 0 (the blade angle) is 8.5, which is much larger than the other inputs. The second input (balance) is 0.65, which is by far the smallest. The sizes of these matter because they factor into the weights and scale with each step.

* **1st weight** - +0.016
* **2nd weight** - +0.001
* **3rd weight** - +0.003

In [3]:
from helpers import w_sum, vect_mat_mul
import numpy as np
# import matplotlib




blade_angle = [8.5, 9.5, 9.9, 9.0] # degrees off-vertical
balance = [0.65, 0.80, 0.80, 0.90] # stance reading, [0, 1]
breath = [1.2, 1.3, 0.5, 1.0] # exhalations per second
clean = [1, 1, 0, 1] # ground truth: clean strike?


# trues[i] = [opens_left, strikes_high, feints] for sensing i
trues_multi = [[0.0, 1.0, 0.0], # sensing 0
               [1.0, 0.0, 0.0], # sensing 1
               [0.0, 0.0, 1.0], # sensing 2 (only feint)
               [1.0, 1.0, 0.0]] # sensing 3




def ele_mul(scalar, vector):
    # from scratch
    # multiply each entry of vector by scalar
    output = [0] * len(vector)
    for i in  range(len(vector)):
        output[i] = scalar * vector[i]


    return output


def gradient_descent_multi(input, weights, true, alpha, iterations):
    # from scratch
    # Returns the final weights, error history, and weight history
    error_history = []
    weight_history = [weights.copy()]
   
    for iter in range(iterations):
        # Predict
        pred = w_sum(input, weights)
        # Compare
        error = (pred - true) ** 2  # MSE
        delta = pred - true


        # Learn
        weight_deltas = ele_mul(delta, input)
        for i in range(len(weights)):
            weights[i] -= alpha * weight_deltas[i]


        weight_history.append(weights.copy())
        error_history.append(error)




    return weights, error_history, weight_history


if __name__ == '__main__':
    #   --- FROM SCRATCH Version ---
    print("\n")
    print("===== FROM SCRATCH VERSION =====")
    print("\n")


    blade_angle = [8.5, 9.5, 9.9, 9.0] # degrees off-vertical
    balance = [0.65, 0.80, 0.80, 0.90] # stance reading, [0, 1]
    breath = [1.2, 1.3, 0.5, 1.0] # exhalations per second
    clean = [1, 1, 0, 1] # ground truth: clean strike?


    sensings = [[blade_angle[i],balance[i], breath[i]] for i in range(4)]


    weights = [0.1, 0.2, -0.1]
    true = 1
    alpha = 0.01
    iterations = 10


    # trues[i] = [opens_left, strikes_high, feints] for sensing i
    trues_multi = [[0.0, 1.0, 0.0], # sensing 0
                   [1.0, 0.0, 0.0], # sensing 1
                   [0.0, 0.0, 1.0], # sensing 2 (only feint)
                   [1.0, 1.0, 0.0]] # sensing 3


    # conduct the gradient descent function which returns these variables
    final_weights, error_history, weight_history = gradient_descent_multi(sensings[0],
                                                                            weights,
                                                                            true,
                                                                            alpha,
                                                                            iterations)
 


    # Print iteration log
    for iter in range(iterations):
        #recalculate the prediction using our weight history
        pred = w_sum(sensings[0], weight_history[iter])
        print(f"Iteration {iter}: Pred - {pred:.6f}, Error - {error_history[iter]:.6f}")


    print(f"Current Weights: {[f'{w:.6f}' for w in final_weights]}")



===== FROM SCRATCH VERSION =====


Iteration 0: Pred - 0.860000, Error - 0.019600
Iteration 1: Pred - 0.963757, Error - 0.001314
Iteration 2: Pred - 0.990618, Error - 0.000088
Iteration 3: Pred - 0.997571, Error - 0.000006
Iteration 4: Pred - 0.999371, Error - 0.000000
Iteration 5: Pred - 0.999837, Error - 0.000000
Iteration 6: Pred - 0.999958, Error - 0.000000
Iteration 7: Pred - 0.999989, Error - 0.000000
Iteration 8: Pred - 0.999997, Error - 0.000000
Iteration 9: Pred - 0.999999, Error - 0.000000
Current Weights: ['0.116057', '0.201228', '-0.097733']


## Numpy Version

In [4]:
sensings_np = np.array(sensings)
weights_np = np.array([0.1, 0.2, -0.1])


for it_np in range(iterations):
    # PREDICT
    # np.dot replaces w_sum
    pred_np = np.dot(sensings_np[0], weights_np)
    # COMPARE
    error_np = (pred_np - true) ** 2
    delta = pred_np - true


    # LEARN
    # replaces the ele_mul and for-loop
    weights_np -= alpha * delta * sensings_np[0]


    print(f"Iter {it_np}: pred={pred_np:.4f} error={error_np:.6f}")


print(f"Current Weights: {[f'{w:.6f}' for w in weights_np]}")

Iter 0: pred=0.8600 error=0.019600
Iter 1: pred=0.9638 error=0.001314
Iter 2: pred=0.9906 error=0.000088
Iter 3: pred=0.9976 error=0.000006
Iter 4: pred=0.9994 error=0.000000
Iter 5: pred=0.9998 error=0.000000
Iter 6: pred=1.0000 error=0.000000
Iter 7: pred=1.0000 error=0.000000
Iter 8: pred=1.0000 error=0.000000
Iter 9: pred=1.0000 error=0.000000
Current Weights: ['0.116057', '0.201228', '-0.097733']


# Part 1b - The Loud Channel

In [5]:
def normalize(channel):
    # returns a list


    # finds the largest value in channel
    biggest = max(channel)


    # returns the channel back but normalized by dividing
    # every input by the biggest value
    return [reading / biggest for reading in channel]


blade_angle = [8.5, 9.5, 9.9, 9.0] # degrees off-vertical
balance = [0.65, 0.80, 0.80, 0.90] # stance reading, [0, 1]
breath = [1.2, 1.3, 0.5, 1.0] # exhalations per second
clean = [1, 1, 0, 1] # ground truth: clean strike?
sensings = [[blade_angle[i],balance[i], breath[i]] for i in range(4)]




# normalize each "channel"
blade_angle_norm = normalize(blade_angle)
balance_norm = normalize(balance)
breath_norm = normalize(breath)


# normalized sensings
sensings_norm = [[blade_angle_norm[i],balance_norm[i], breath_norm[i]] for i in range(4)]


print("-- Sensing 0 Normalized --")
for value in sensings_norm[0]:
    print(f"{value:.4f}")


from part1_multi_input import gradient_descent_multi


weights = [0.1, 0.2, -0.1]
true = 1
alpha_sc = 0.1
alpha_raw = 0.01
iterations = 20


_, error_history_raw, _ = gradient_descent_multi(sensings[0],
                                                 weights,                                                                      
                                                 true,
                                                 alpha_raw,
                                                 iterations)


_, error_history_norm, _ = gradient_descent_multi(sensings_norm[0],
                                                 weights,                                                                      
                                                 true,
                                                 alpha_sc,
                                                 iterations)


print("")
for it in range(iterations):
    print(f"Iteration {it}: Raw Error: {error_history_raw[it]:.6f}, Scaled Error: {error_history_norm[it]:.6f}")


-- Sensing 0 Normalized --
0.8586
0.7222
0.9231

Iteration 0: Raw Error: 0.019600, Scaled Error: 0.714430
Iteration 1: Raw Error: 0.001314, Scaled Error: 0.444652
Iteration 2: Raw Error: 0.000088, Scaled Error: 0.276746
Iteration 3: Raw Error: 0.000006, Scaled Error: 0.172243
Iteration 4: Raw Error: 0.000000, Scaled Error: 0.107202
Iteration 5: Raw Error: 0.000000, Scaled Error: 0.066721
Iteration 6: Raw Error: 0.000000, Scaled Error: 0.041526
Iteration 7: Raw Error: 0.000000, Scaled Error: 0.025846
Iteration 8: Raw Error: 0.000000, Scaled Error: 0.016086
Iteration 9: Raw Error: 0.000000, Scaled Error: 0.010012
Iteration 10: Raw Error: 0.000000, Scaled Error: 0.006231
Iteration 11: Raw Error: 0.000000, Scaled Error: 0.003878
Iteration 12: Raw Error: 0.000000, Scaled Error: 0.002414
Iteration 13: Raw Error: 0.000000, Scaled Error: 0.001502
Iteration 14: Raw Error: 0.000000, Scaled Error: 0.000935
Iteration 15: Raw Error: 0.000000, Scaled Error: 0.000582
Iteration 16: Raw Error: 0.000000

### Comment Section

**(a) Raw inputs at alpha = 0.1 diverge; scaled inputs at the same alpha do not. Explain it in terms of `weight_delta = delta * input`.**

The size of the weight update is scaled by the input because weight delta is alpha * delta * input. THis means that for the raw input with the large input and a larger input cause it to overshoot its prediction and it will continue overshooting, causing divergence. As for scaled data data, the same alpha gives a controlled step because every input value is less than, causing it to not diverge.

**(b) Your error after scaling is larger than the raw alpha = 0.01 run at every iteration, not just the first. Why is that not a regression? What are you actually comparing when you say one run is better than the other?**

The runs are not necessarily the same. The raw run looks better because it happened to be a good fit for that scale, while the scaled run's alpha is a generic value that is not tuned. If the numbers were to change on the raw run, it is highly likely that it would not do well. Compared to the scaled run, you could change the alpha and it would do better because the input stays in the small range.

**(c) Week 2 also gave you min-max. Applied to `blade_angle`, what does it make sensing 0's first entry? What then happens to that weight, and which later part of this assignment does that resemble?**

Sensing 0's first entry would be 0. So then when it tries to udpate the weight, it simply doesn't and does not move. This resembles dead neurons or freezing weights.

# Part 2 - One In, Many Out

In [6]:

import numpy as np

# ---------------------------------------------------------------
# Part 1: from-scratch gradient descent, one input -> three outputs
# ---------------------------------------------------------------
def gradient_descent_outputs(input, weights, trues, alpha, iterations):
    """
    input   : scalar
    weights : length-3 list of starting weights
    trues   : length-3 list of target outputs
    alpha   : learning rate
    iterations : number of training steps

    Returns (final_weights, error_history, weight_history)
      error_history  -> list of per-iteration mean squared error across the 3 outputs
      weight_history -> list of the 3 weights after each iteration
    """
    weights = list(weights)
    error_history = []
    weight_history = []

    for _ in range(iterations):
        # one shared input drives three independent predictions
        preds = [w * input for w in weights]

        # per-output raw error (pred - true), and MSE across the three outputs
        deltas = [p - t for p, t in zip(preds, trues)]
        mse = sum(d ** 2 for d in deltas) / len(deltas)
        error_history.append(mse)

        # each weight's update is scaled by the SAME input
        weight_deltas = [d * input for d in deltas]
        weights = [w - alpha * wd for w, wd in zip(weights, weight_deltas)]

        weight_history.append(list(weights))

    return weights, error_history, weight_history


# ---------------------------------------------------------------
# Part 2: train and print a clean iteration log
# ---------------------------------------------------------------
balance = [0.65, 0.20, 0.90]           # only balance[0] is used here
trues_multi = [[0.0, 1.0, 0.0], [1.0, 0.0, 1.0], [0.0, 0.0, 1.0]]

input_val = balance[0]
start_weights = [0.3, 0.2, 0.9]
trues = trues_multi[0]
alpha = 0.1
iterations = 20

final_w, err_hist, w_hist = gradient_descent_outputs(
    input_val, start_weights, trues, alpha, iterations
)

print("From-scratch training log")
print(f"{'iter':>4}  {'w0':>8} {'w1':>8} {'w2':>8}   {'mse':>10}")
for i, (w, e) in enumerate(zip(w_hist, err_hist), start=1):
    print(f"{i:>4}  {w[0]:8.5f} {w[1]:8.5f} {w[2]:8.5f}   {e:10.6f}")

print("\nFinal weights:", [round(w, 5) for w in final_w])
print("Final MSE:", round(err_hist[-1], 6))

From-scratch training log
iter        w0       w1       w2          mse
   1   0.28732  0.25655  0.86198     0.379050
   2   0.27519  0.31071  0.82556     0.347697
   3   0.26356  0.36258  0.79068     0.318937
   4   0.25242  0.41226  0.75727     0.292556
   5   0.24176  0.45985  0.72528     0.268358
   6   0.23154  0.50542  0.69463     0.246160
   7   0.22176  0.54906  0.66528     0.225799
   8   0.21239  0.59087  0.63718     0.207122
   9   0.20342  0.63090  0.61026     0.189990
  10   0.19482  0.66925  0.58447     0.174275
  11   0.18659  0.70597  0.55978     0.159860
  12   0.17871  0.74114  0.53613     0.146637
  13   0.17116  0.77483  0.51348     0.134508
  14   0.16393  0.80709  0.49178     0.123382
  15   0.15700  0.83799  0.47100     0.113177
  16   0.15037  0.86759  0.45110     0.103815
  17   0.14402  0.89593  0.43205     0.095228
  18   0.13793  0.92308  0.41379     0.087351
  19   0.13210  0.94908  0.39631     0.080126
  20   0.12652  0.97398  0.37956     0.073499

Final w

## Numpy Version

In [7]:
input_val = balance[0]
start_weights = [0.3, 0.2, 0.9]
trues = trues_multi[0]
alpha = 0.1
iterations = 20



def gradient_descent_outputs_np(input, weights, trues, alpha, iterations):
    w = np.array(weights, dtype=float)
    t = np.array(trues, dtype=float)
    error_history = []
    weight_history = []

    for _ in range(iterations):
        preds = w * input         # broadcast: scalar input, 3 weights
        deltas = preds - t
        mse = np.mean(deltas ** 2)
        error_history.append(mse)

        w = w - alpha * (deltas * input)
        weight_history.append(w.copy())

    return w, error_history, weight_history

np_final_w, np_err_hist, np_w_hist = gradient_descent_outputs_np(
    input_val, start_weights, trues, alpha, iterations
)

# parity check
w_match = np.allclose(final_w, np_final_w)
e_match = np.allclose(err_hist, np_err_hist)
print("\nNumPy parity check")
print("Final weights match:", w_match, "->", np_final_w)
print("Error histories match:", e_match)


NumPy parity check
Final weights match: True -> [0.12652154 0.97398082 0.37956462]
Error histories match: True


In [8]:
# ---------------------------------------------------------------
# Part 4: first-iteration vs last-iteration delta ratios
# ---------------------------------------------------------------
def per_output_deltas(input, weights, trues):
    return [w * input - t for w, t in zip(weights, trues)]

first_deltas = per_output_deltas(input_val, start_weights, trues)
last_deltas  = per_output_deltas(input_val, final_w, trues)

print("\nDelta comparison (iteration 1 vs iteration 20)")
for i in range(3):
    ratio = last_deltas[i] / first_deltas[i]
    print(f"output {i}: first={first_deltas[i]:+.5f}  last={last_deltas[i]:+.5f}  ratio={ratio:.5f}")


Delta comparison (iteration 1 vs iteration 20)
output 0: first=+0.19500  last=+0.08224  ratio=0.42174
output 1: first=-0.87000  last=-0.36691  ratio=0.42174
output 2: first=+0.58500  last=+0.24672  ratio=0.42174


**Delta Ratio Analysis**

All three outputs' deltas shrink by the same factor (**0.42174**) each iteration because that decay rate depends only on the shared input and alpha, not on each output's target. 

* **Closest Prediction:** Output 0 ends up closest to its goal purely because it started closest, not because it converged faster.
* **Convergence Rate:** Output 1 is still notably off after 20 iterations.
* **Key Takeaway:** A single shared input forces every output to learn at the same rate regardless of how far off it starts.

# Part 3 - Many In, Many Out

In [9]:
from helpers import vect_mat_mul
import numpy as np

def outer_prod(deltas, inputs):
    output = []

    for i in range(len(deltas)):
        row = []

        for j in range(len(inputs)):
            row.append(deltas[i] * inputs[j])

        output.append(row)

    return output


def gradient_descent_full(input, weights, trues, alpha, iterations):
    # Copy each row so the original starting weights are not changed
    weights = [row.copy() for row in weights]

    error_history = []
    weight_history = []

    for iteration in range(iterations):

        # Make one prediction for each output
        pred = vect_mat_mul(input, weights)

        # Calculate how far each prediction is from its target
        deltas = []

        for i in range(len(pred)):
            deltas.append(pred[i] - trues[i])

        # Mean squared error across all three outputs
        error = 0

        for i in range(len(pred)):
            error += (pred[i] - trues[i]) ** 2

        error /= len(pred)

        # Each output delta is multiplied by each input
        # This creates a 3x3 matrix of weight deltas
        weight_deltas = outer_prod(deltas, input)

        # Update every weight
        for i in range(len(weights)):
            for j in range(len(weights[i])):
                weights[i][j] -= alpha * weight_deltas[i][j]

        error_history.append(error)

        # Save a copy of the entire weight matrix
        weight_history.append([row.copy() for row in weights])

        print(
            f"Iteration {iteration}: "
            f"pred={pred}, "
            f"error={error}, "
            f"weights={weights}"
        )

    return weights, error_history, weight_history



## Numpy Version

In [10]:
# --- NumPy version ---

def gradient_descent_full_numpy(input, weights, trues, alpha, iterations):
    input = np.array(input, dtype=float)
    weights = np.array(weights, dtype=float)
    trues = np.array(trues, dtype=float)

    error_history = []
    weight_history = []

    for iteration in range(iterations):

        # Same as vect_mat_mul()
        pred = np.dot(weights, input)

        # Prediction - target for all three outputs at once
        deltas = pred - trues

        # Mean squared error across the three outputs
        error = np.mean(np.square(deltas))

        # Creates the same 3x3 weight delta matrix as outer_prod()
        weight_deltas = np.outer(deltas, input)

        # Update every weight at once
        weights -= alpha * weight_deltas

        error_history.append(error)
        weight_history.append(weights.copy())

    return weights, error_history, weight_history


In [11]:
if __name__ == "__main__":
            # blade balance breath
    input = [8.5, 0.65, 1.2]

    weights = [
        [0.1, 0.1, -0.3],   # opens_left
        [0.1, 0.2, 0.0],    # strikes_high
        [0.0, 1.3, 0.1]     # feints
    ]

    trues = [0.0, 1.0, 0.0]

    alpha = 0.01
    iterations = 15

    final_weights, errors, weight_history = gradient_descent_full(
        input,
        weights,
        trues,
        alpha,
        iterations
    )

    print("\nFinal weight matrix:")

    for row in final_weights:
        print(row)

    numpy_weights, numpy_errors, numpy_weight_history = gradient_descent_full_numpy(
    input,
    weights,
    trues,
    alpha,
    iterations
    )

    print("\nNumPy final weight matrix:")
    print(numpy_weights)

    print(
        "\nFrom-scratch and NumPy match:",
        np.allclose(final_weights, numpy_weights, atol=1e-10)
    ) 

Iteration 0: pred=[0.555, 0.9800000000000001, 0.9650000000000001], error=0.41321666666666673, weights=[[0.052825000000000004, 0.0963925, -0.30666], [0.1017, 0.20013, 0.00023999999999999887], [-0.082025, 1.2937275000000001, 0.08842]]
Iteration 1: pred=[0.14367562500000008, 0.9948224999999999, 0.24981437500000003], error=0.02769223789401043, weights=[[0.040612571874999996, 0.09545860843750001, -0.3083841075], [0.1021400875, 0.20016365375, 0.0003021299999999995], [-0.103259221875, 1.2921037065625, 0.0854222275]]
Iteration 2: pred=[0.037194027421875, 0.9986596746875, 0.06467069632812512], error=0.0018558303704556977, weights=[[0.03745107954414062, 0.09521684725925782, -0.3088304358290625], [0.1022540151515625, 0.20017236586453124, 0.00031821390374999885], [-0.10875623106289063, 1.2916833470363673, 0.0846461791440625]]
Iteration 3: pred=[0.009628603848837847, 0.9996530232847265, 0.0167416265119434], error=0.00012437082106140103, weights=[[0.0366326482169894, 0.09515426133424038, -0.30894597

### Print the final weight matrix. Which weights changed substantially? Which barely moved? Explain using both factors in `weight_deltas[i][j] = deltas[i] * input[j]` — movement across a row is driven by input size, movement between rows by that output's delta. Why did the `strikes_high` row barely move at all?

Within each row, the blade_angle weight changed the most. This is because the input for the blade angle was the highest at 8.5. Because `weight_delta = delta * input`, that high input made the weight delta higher than the other inputs' weight_deltas. Therefore the blade_angle weight changed the most.

The feints row changed the most because it originally had the highest delta. The prediction was .965 while the target was 0.0.

The `strikes_high` row changed the least because its prediction started at 0.98 and the target was 1.0. Its delta was very small so all the weight deltas for that row were also small.

# Part 4 - Freezing the Inner Gaze

In [12]:
def ele_mul(scalar, vector):
    return [scalar * x for x in vector]


def gradient_descent_frozen(input_vec, weights, true, alpha, iterations, frozen):
    weights = list(weights)
    error_history = []
    weight_history = [list(weights)]

    for _ in range(iterations):
        pred = sum(i * w for i, w in zip(input_vec, weights))
        error = (pred - true) ** 2
        delta = pred - true
        error_history.append(error)

        raw_weight_deltas = ele_mul(delta, input_vec)

        # Freeze specified weights by setting delta to 0
        weight_deltas = [
            0.0 if i in frozen else raw_weight_deltas[i]
            for i in range(len(weights))
        ]

        # Update unfrozen weights
        weights = [
            w - alpha * wd for w, wd in zip(weights, weight_deltas)
        ]
        weight_history.append(list(weights))

    return weights, error_history, weight_history


if __name__ == "__main__":
    sensing_0 = [8.5, 0.65, 1.2]
    init_weights = [0.1, 0.2, -0.1]
    target = 1.0
    num_iters = 5

    configs = [
        {"name": "Baseline (Unfrozen)", "frozen": [], "alpha": 0.01},
        {"name": "Only Balance Learns", "frozen": [0, 2], "alpha": 0.3},
        {"name": "Only Breath Learns", "frozen": [0, 1], "alpha": 0.3},
    ]

    for cfg in configs:
        print(f"=== {cfg['name']} (frozen={cfg['frozen']}, alpha={cfg['alpha']}) ===")
        final_w, errs, w_hist = gradient_descent_frozen(
            sensing_0, init_weights, target, cfg["alpha"], num_iters, cfg["frozen"]
        )

        for it in range(num_iters):
            pred = sum(i * w for i, w in zip(sensing_0, w_hist[it]))
            print(
                f"Iter {it}: Pred = {pred:.4f}, Error = {errs[it]:.6f}, "
                f"Weights = {[round(w, 4) for w in w_hist[it]]}"
            )
        print(f"Final Weights: {[round(w, 4) for w in final_w]}\n")

=== Baseline (Unfrozen) (frozen=[], alpha=0.01) ===
Iter 0: Pred = 0.8600, Error = 0.019600, Weights = [0.1, 0.2, -0.1]
Iter 1: Pred = 0.9638, Error = 0.001314, Weights = [0.1119, 0.2009, -0.0983]
Iter 2: Pred = 0.9906, Error = 0.000088, Weights = [0.115, 0.2011, -0.0979]
Iter 3: Pred = 0.9976, Error = 0.000006, Weights = [0.1158, 0.2012, -0.0978]
Iter 4: Pred = 0.9994, Error = 0.000000, Weights = [0.116, 0.2012, -0.0977]
Final Weights: [0.116, 0.2012, -0.0977]

=== Only Balance Learns (frozen=[0, 2], alpha=0.3) ===
Iter 0: Pred = 0.8600, Error = 0.019600, Weights = [0.1, 0.2, -0.1]
Iter 1: Pred = 0.8777, Error = 0.014946, Weights = [0.1, 0.2273, -0.1]
Iter 2: Pred = 0.8932, Error = 0.011398, Weights = [0.1, 0.2511, -0.1]
Iter 3: Pred = 0.9068, Error = 0.008691, Weights = [0.1, 0.272, -0.1]
Iter 4: Pred = 0.9186, Error = 0.006628, Weights = [0.1, 0.2901, -0.1]
Final Weights: [0.1, 0.306, -0.1]

=== Only Breath Learns (frozen=[0, 1], alpha=0.3) ===
Iter 0: Pred = 0.8600, Error = 0.01960

#### In each frozen run, identify which weight is doing all of the correcting, and explain in your own words why a frozen weight forces the others to compensate. (Thursday's slide called this "absorbing the slack.") Then answer the learning-rate question. Week 3 showed that a large input tightens the error curve; here is that idea as a number. Each iteration multiplies the miss by $1 - \alpha \cdot S$, where $S$ is the sum of $\text{input}[i]^2$ taken over the weights that are *still free* — a frozen weight contributes nothing to $S$, because its `weight_delta` is zeroed before the update. With every weight free, $S = 8.5^2 + 0.65^2 + 1.2^2 = 74.1125$; with indices 0 and 2 frozen, $S = 0.65^2 = 0.4225$; with indices 0 and 1 frozen, $S = 1.2^2 = 1.44$. Compute $1 - \alpha \cdot S$ for all three of your runs, check each one against the ratio of consecutive misses in your own log, and use those three numbers to explain why the frozen runs are safe at an alpha thirty times the one the baseline is given.

---

**COMMENT BLOCK: FREEZING ANALYSIS**

**1. Identification of Compensating Weights & Absorbing Slack:**

* **Baseline (`frozen=[]`):** All three weights update concurrently based on their input magnitudes.
* **Configuration 1 (`frozen=[0, 2]`):** The 'balance' weight (index 1) does ALL the correcting. Because `blade_angle` and `breath` are locked and silent, the `balance` weight is forced to absorb all the residual error ("slack") to drive the prediction toward 1.0.
* **Configuration 2 (`frozen=[0, 1]`):** The 'breath' weight (index 2) does ALL the correcting for the same reason.

---

**2. Mathematical Explanation of Learning-Rate Tolerance:**

Each iteration multiplies the prediction miss (delta) by the factor $(1 - \alpha \cdot S)$, where $S = \sum \text{input}[i]^2$ evaluated **ONLY** over active (unfrozen) inputs.

* **Baseline (All free, $\alpha = 0.01$):**
  $$S = 8.5^2 + 0.65^2 + 1.2^2 = 72.25 + 0.4225 + 1.44 = 74.1125$$
  $$\text{Scaling factor} = 1 - (0.01 \cdot 74.1125) = 1 - 0.741125 = 0.258875$$
  *(The miss shrinks stably by ~74% per step).*
  
  *\*Note: If $\alpha = 0.3$ were used here, $1 - (0.3 \cdot 74.1125) = -21.23375$, causing extreme divergence and flipping signs.*

* **Balance Only Free (`frozen=[0, 2]`, $\alpha = 0.3$):**
  $$S = 0.65^2 = 0.4225$$
  $$\text{Scaling factor} = 1 - (0.3 \cdot 0.4225) = 1 - 0.12675 = 0.87325$$
  *(The miss shrinks stably by ~12.7% per step).*

* **Breath Only Free (`frozen=[0, 1]`, $\alpha = 0.3$):**
  $$S = 1.2^2 = 1.44$$
  $$\text{Scaling factor} = 1 - (0.3 \cdot 1.44) = 1 - 0.432 = 0.568$$
  *(The miss shrinks stably by ~43.2% per step).*

---

**Conclusion:** 
Freezing index 0 removes the massive `blade_angle` input (8.5) from $S$, dropping $S$ from 74.1125 to $\le 1.44$. Because the stability limit requires $|1 - \alpha \cdot S| < 1$, removing the large input allows us to safely scale $\alpha$ up by 30x without overshooting or exploding.

# Part 5 - Watching the Weights

## Trajectory and Plotting Analysis

**Question:** In Figure 2, how does a free weight’s trajectory change when the weights beside it are frozen? Which of Figure 1’s three lines is steepest, and does that match the prediction you made in Part 1’s comment block? Two of Figure 1’s lines look flat. Are they? What would you plot instead to find out?

---

**Free Weight Trajectory (Figure 2)**

In Figure 2, freezing the weights beside a free weight makes that free weight move further and faster than it does in the unfrozen baseline: over 10 iterations, frozen balance moves +0.00893 vs. only +0.00123 unfrozen, and frozen breath moves +0.01575 vs. only +0.00227 unfrozen -- roughly 7x more in both cases. 

In the unfrozen run, `blade_angle` (the largest input) does most of the work closing the gap between prediction and target, so delta shrinks quickly and every weight's per-step update shrinks with it. With `blade_angle` frozen, it can't help close that gap, so delta stays larger for longer and whichever single weight is still free has to absorb all of that lingering correction alone.

---

**Steepest Line Comparison (Figure 1)**

`blade_angle` is the steepest line in Figure 1 (`0.1` -> `0.116`, a change of about **+0.016**), far more than balance (**+0.001**) or breath (**+0.002**). That matches Part 1's own comment block, which predicted `blade_angle` would change the most because its input (`8.5`) is so much larger than balance's (`0.65`) or breath's (`1.2`) -- `weight_delta = delta * input`, so the same shared delta gets scaled up far more for `blade_angle`.

---

**Apparent Flat Lines & Plotting Alternatives**

The balance and breath lines in Figure 1 are **NOT** actually flat -- they do move (by **+0.001** and **+0.002** respectively) -- they only look flat because the y-axis has to span `blade_angle`'s much larger swing, which squashes their movement down to a sliver. 

To see their real shape, plot balance and breath on their own separate axes (e.g. one subplot per weight, or a zoomed-in plot restricted to their own value range) instead of sharing one y-axis with `blade_angle`.

In [13]:
import matplotlib  # type: ignore[reportMissingModuleSource]
matplotlib.use("Agg")  # render to a file; no display needed
import matplotlib.pyplot as plt  # type: ignore[reportMissingModuleSource]

from part1_multi_input import gradient_descent_multi
from part4_freeze import gradient_descent_frozen

blade_angle, balance, breath = 8.5, 0.65, 1.2
input_val = [blade_angle, balance, breath]
start_weights = [0.1, 0.2, -0.1]
true = 1.0
alpha = 0.01
iterations = 10


# --- Figure 1: Part 1's weights over time ---
weights, errors, weight_history = gradient_descent_multi(
    list(input_val), list(start_weights), true, alpha, iterations
)

for j, name in enumerate(["blade_angle", "balance", "breath"]):
    plt.plot([w[j] for w in weight_history], label=name)
plt.xlabel("iteration")
plt.ylabel("weight value")
plt.title("Part 1 weights, alpha = 0.01")
plt.legend()
plt.savefig("fig1_weights.png", dpi=150)
plt.close()  # start a clean figure for the next plot


# --- Figure 2: frozen vs. unfrozen ---
_, _, balance_frozen_hist = gradient_descent_frozen(
    list(input_val), list(start_weights), true, alpha, iterations, frozen=[0, 2]
)
_, _, breath_frozen_hist = gradient_descent_frozen(
    list(input_val), list(start_weights), true, alpha, iterations, frozen=[0, 1]
)

plt.plot([w[1] for w in balance_frozen_hist], label="balance, frozen=[0, 2], alpha=0.01")
plt.plot([w[1] for w in weight_history], label="balance, unfrozen, alpha=0.01")
plt.plot([w[2] for w in breath_frozen_hist], label="breath, frozen=[0, 1], alpha=0.01")
plt.plot([w[2] for w in weight_history], label="breath, unfrozen, alpha=0.01")
plt.xlabel("iteration")
plt.ylabel("weight value")
plt.title("Part 4 frozen vs. unfrozen weight trajectories")
plt.legend()
plt.savefig("fig2_frozen.png", dpi=150)
plt.close()

print("Saved fig1_weights.png and fig2_frozen.png")

Saved fig1_weights.png and fig2_frozen.png


![Figure 1 Weights](fig1_weights.png)

![Figure 2 Frozen](fig2_frozen.png)

# Part 6 - Unit Tests

In [22]:
# week4/test_multi.py -- Unit tests for Week 4 multi-input/output gradient descent
import numpy as np
from helpers import normalize
from part1_multi_input import gradient_descent_multi
from part2_multi_output import gradient_descent_outputs
from part3_multi_in_multi_out import gradient_descent_full, gradient_descent_full_numpy, outer_prod
from part4_freeze import gradient_descent_frozen


# 1. Normalization
def test_normalize_scales_and_preserves_input():
    """Verify normalize leaves max reading at 1.0, scales rest, and does not mutate input."""
    original = [2.0, 4.0, 8.0]
    original_copy = list(original)

    result = normalize(original)

    assert result == [0.25, 0.5, 1.0], f"Expected [0.25, 0.5, 1.0], got {result}"
    assert original == original_copy, "normalize modified the original list"


# 2. Multi-Input GD
def test_multi_input_gd_reduces_error():
    """Verify gradient_descent_multi reduces error over iterations."""
    sensing = [8.5, 0.65, 1.2]
    weights = [0.1, 0.2, -0.1]
    true = 1.0
    alpha = 0.001
    iterations = 10

    _, error_history, _ = gradient_descent_multi(
        sensing, weights, true, alpha, iterations
    )

    assert error_history[-1] < error_history[0], (
        f"Final error ({error_history[-1]:.6f}) should be less than initial ({error_history[0]:.6f})"
    )


def test_multi_input_gd_prediction_close_to_goal():
    """Verify final prediction is close to goal for an easy sensing input."""
    sensing = [1.0, 1.0, 1.0]
    weights = [0.1, 0.1, 0.1]
    true = 1.0
    alpha = 0.1
    iterations = 50

    final_w, _, _ = gradient_descent_multi(
        sensing, weights, true, alpha, iterations
    )
    final_pred = sum(s * w for s, w in zip(sensing, final_w))

    assert abs(final_pred - true) < 1e-3, (
        f"Final prediction ({final_pred:.4f}) is not close to target ({true})"
    )


# 3. Multi-Output GD
def test_multi_output_gd_predictions_move_toward_target():
    """Verify each output's prediction moves closer to its target."""
    input_val = 0.65
    weights = [0.3, 0.2, 0.9]
    trues = [0.0, 1.0, 0.0]
    alpha = 0.1
    iterations = 20

    initial_errors = [abs(w * input_val - t) for w, t in zip(weights, trues)]
    final_w, _, _ = gradient_descent_outputs(
        input_val, weights, trues, alpha, iterations
    )
    final_errors = [abs(w * input_val - t) for w, t in zip(final_w, trues)]

    for i in range(len(trues)):
        assert final_errors[i] < initial_errors[i], (
            f"Output {i} error did not decrease: initial {initial_errors[i]}, final {final_errors[i]}"
        )


def test_multi_output_gd_weights_change_unless_pred_equals_target():
    """Verify all weights change unless prediction equals target (zero delta)."""
    input_val = 2.0
    # w0 * input_val = 0.5 * 2.0 = 1.0 -> target = 1.0 (zero delta)
    weights = [0.5, 0.2, 0.9]
    trues = [1.0, 0.0, 0.0]
    alpha = 0.05
    iterations = 5

    final_w, _, _ = gradient_descent_outputs(
        input_val, weights, trues, alpha, iterations
    )

    assert abs(final_w[0] - weights[0]) < 1e-12, "Weight 0 changed despite zero delta!"
    assert abs(final_w[1] - weights[1]) > 1e-6, "Weight 1 should have changed."
    assert abs(final_w[2] - weights[2]) > 1e-6, "Weight 2 should have changed."


# 4. Outer Product
def test_outer_product_matrix():
    """Verify outer_prod([1, 2], [3, 4, 5]) produces expected 2x3 matrix."""
    result = outer_prod([1, 2], [3, 4, 5])
    expected = [[3, 4, 5], [6, 8, 10]]
    assert result == expected, f"Expected {expected}, got {result}"


# 5. Freezing
from part4_freeze import gradient_descent_frozen  # Updated name


def test_freezing_frozen_weights_do_not_change():
    """Verify weight indices in frozen remain unchanged across iterations."""
    sensing = [8.5, 0.65, 1.2]
    weights = [0.1, 0.2, -0.1]
    true = 1.0
    alpha = 0.01
    iterations = 10
    frozen = [0, 2]  # Freeze 1st and 3rd weights

    final_w, _, _ = gradient_descent_frozen(
        sensing, weights, true, alpha, iterations, frozen
    )

    assert final_w[0] == weights[0], "Frozen weight at index 0 changed!"
    assert final_w[2] == weights[2], "Frozen weight at index 2 changed!"


def test_freezing_unfrozen_weights_still_change():
    """Verify weights NOT in frozen continue to update."""
    sensing = [8.5, 0.65, 1.2]
    weights = [0.1, 0.2, -0.1]
    true = 1.0
    alpha = 0.01
    iterations = 10
    frozen = [0, 2]  # Index 1 is unfrozen

    final_w, _, _ = gradient_descent_frozen(
        sensing, weights, true, alpha, iterations, frozen
    )

    assert abs(final_w[1] - weights[1]) > 1e-6, "Unfrozen weight at index 1 did not change!"

# 6. From-Scratch vs. NumPy Agreement
def test_from_scratch_vs_numpy_agreement():
    """Verify full GD pure-Python and NumPy implementations match within 1e-10."""
    inputs = [8.5, 0.65, 1.2]
    weights = [[0.1, 0.1, -0.3], [0.1, 0.2, 0.0], [0.0, 1.3, 0.1]]
    trues = [0.0, 1.0, 0.0]
    alpha = 0.01
    iterations = 15

    py_w, py_err, _ = gradient_descent_full(inputs, weights, trues, alpha, iterations)
    np_w, np_err, _ = gradient_descent_full_numpy(inputs, weights, trues, alpha, iterations)

    for r_py, r_np in zip(py_w, np_w):
        for w_py, w_np in zip(r_py, r_np):
            assert abs(w_py - w_np) < 1e-10, f"Weight mismatch: {w_py} vs {w_np}"

    for e_py, e_np in zip(py_err, np_err):
        assert abs(e_py - e_np) < 1e-10, f"MSE mismatch: {e_py} vs {e_np}"


# Test Runner
if __name__ == "__main__":
    tests = [
        name
        for name, value in globals().items()
        if name.startswith("test_") and callable(value)
    ]
    for test_name in sorted(tests):
        test_func = globals()[test_name]
        try:
            test_func()
            print(f"PASS: {test_name}")
        except AssertionError as e:
            print(f"FAIL: {test_name} -- {e}")

PASS: test_freezing_frozen_weights_do_not_change
PASS: test_freezing_unfrozen_weights_still_change
Iteration 0: pred=[0.555, 0.9800000000000001, 0.9650000000000001], error=0.41321666666666673, weights=[[0.052825000000000004, 0.0963925, -0.30666], [0.1017, 0.20013, 0.00023999999999999887], [-0.082025, 1.2937275000000001, 0.08842]]
Iteration 1: pred=[0.14367562500000008, 0.9948224999999999, 0.24981437500000003], error=0.02769223789401043, weights=[[0.040612571874999996, 0.09545860843750001, -0.3083841075], [0.1021400875, 0.20016365375, 0.0003021299999999995], [-0.103259221875, 1.2921037065625, 0.0854222275]]
Iteration 2: pred=[0.037194027421875, 0.9986596746875, 0.06467069632812512], error=0.0018558303704556977, weights=[[0.03745107954414062, 0.09521684725925782, -0.3088304358290625], [0.1022540151515625, 0.20017236586453124, 0.00031821390374999885], [-0.10875623106289063, 1.2916833470363673, 0.0846461791440625]]
Iteration 3: pred=[0.009628603848837847, 0.9996530232847265, 0.016741626511

TypeError: 'str' object is not callable